# Example Analysis Pipeline

This notebook demonstrates a simple pipeline using the `community_collection` package.

## 1. Sample data

In [ ]:
import pandas as pd
sample_messages = [
    {"id": 1, "timestamp": "2024-01-01", "content": "Hello world from OpenAI!", "embeds": []},
    {"id": 2, "timestamp": "2024-01-02", "content": "Codex generates code and text.", "embeds": []},
    {"id": 3, "timestamp": "2024-01-03", "content": "Artificial intelligence is transforming research.", "embeds": []}
]
df = pd.DataFrame(sample_messages)
df

## 2. Parse messages

In [ ]:
from community_collection import parse_message, create_combined_content
parsed = [parse_message(m) for m in sample_messages]
messages_df = pd.DataFrame(parsed)
messages_df = create_combined_content(messages_df)
messages_df

## 3. Add NLP features

In [ ]:
from community_collection import add_ner_columns, add_nounchunk_columns
messages_df = add_ner_columns(messages_df, 'combined_content', model='en_core_web_sm')
messages_df = add_nounchunk_columns(messages_df, 'combined_content', model='en_core_web_sm')
messages_df[['combined_content_entities', 'combined_content_noun_chunks']].head()

## 4. Chunk and vectorise

In [ ]:
from community_collection import chunk, vectorise
chunks = chunk(messages_df['combined_content'].tolist(), model='intfloat/multilingual-e5-small', size=50)
chunk_vectors = vectorise(chunks, model='intfloat/multilingual-e5-small', batch_size=8, progress=False)
chunk_vectors[:2]

## 5. Reduce dimensions and cluster

In [ ]:
from community_collection import reduce_vectors, cluster
cluster_vecs, map_vecs = reduce_vectors(chunk_vectors)
labels = cluster(cluster_vecs)
labels

## 6. Topic modelling

In [ ]:
from community_collection import use_bertopic_with_custom_vectors
model, topics = use_bertopic_with_custom_vectors(chunk_vectors, messages_df['combined_content'].tolist(), n_topics=3)
print(topics)

This pipeline processes a small set of messages, generates embeddings, clusters them and assigns topics using BERTopic.